# Journal technical validation: 512-bit integration, N=128, and WAN

This notebook produces the additional evidence requested for the
journal version of the SLE paper. It addresses three concrete points:

| Point | Executable evidence |
|---|---|
| 2 | One integrated five-party MPC run with **128 identities**, **512-bit biometric roots**, and **64-bit locators**. The same circuit performs SLE transition verification and secure 1:N Hamming/ArgMin. |
| 3 | Three repeated **single-host emulated-WAN** measurements. Linux `tc/netem` is used when available; otherwise a verified transport-level delay is injected into every logical MP-SPDZ message. Local and WAN measurements use the same bytecode, inputs, and five-party Shamir backend. |
| 6 | Transition scaling for `N = 8, 16, 32, 64, 128`, including both SLE and the generic secret pointer-join baseline. |

The biometric experiment and the transition microbenchmark are kept
separate in the output tables. This prevents a 512-bit biometric root
from being confused with locator width.

**Hardware:** GPU is used only for FaceNet extraction. MP-SPDZ runs on
CPU. The paper profile is intentionally longer than a smoke test.


In [ ]:
# CELL 1. Reproducible experiment controls
# Use RUN_PROFILE='smoke' once if this is a new Colab runtime. The
# journal measurements require RUN_PROFILE='paper'.

from pathlib import Path

RUN_PROFILE = 'paper'          # 'smoke' or 'paper'
SEED = 20260804
MP_SPDZ_REVISION = '9d809599ea6ce627216a389ca7d984fbb75d0cb9'

PLAYERS = 5
THRESHOLD = 2
EPOCH = 43

# Point 2: integrated biometric configuration.
BIOMETRIC_IDENTITIES = 16 if RUN_PROFILE == 'smoke' else 128
BIOMETRIC_ROOT_BITS = 512
BIOMETRIC_LOCATOR_BITS = 64
BIOMETRIC_BLOCK_BITS = 4
IMAGES_PER_ID = 4
GALLERY_IMAGES_PER_ID = 2
INTEGRATED_QUERY_ROWS = [0] if RUN_PROFILE == 'smoke' else [0, 1, 2]
CLOSED_SET_THRESHOLD = BIOMETRIC_ROOT_BITS

# A precomputed NPZ can be used to skip CelebA and FaceNet. Required:
# gallery_roots, probe_roots, labels, probe_labels.
BIOMETRIC_NPZ = None
USE_SYNTHETIC_BIOMETRIC_SMOKE = False

# Point 6: transition scaling. This is deliberately a 32-bit synthetic
# root benchmark because it isolates epoch-transition cost. Locator
# width is measured separately below.
SCALING_N = [8, 16, 32, 64, 128]
SCALING_ROOT_BITS = 32
SCALING_BLOCK_BITS = 4
SCALING_LOCATOR_BITS = 32
SCALING_REPEATS = 1 if RUN_PROFILE == 'smoke' else 3
RUN_POINTER_JOIN = True

# Point 2 also requests a realistic locator-width point.
WIDTH_TEST_N = 16 if RUN_PROFILE == 'smoke' else 32
LOCATOR_WIDTHS = [32, 64]
WIDTH_REPEATS = 1 if RUN_PROFILE == 'smoke' else 3

# Point 3: verified latency emulation. A smaller N keeps the WAN
# experiment practical because an interactive MPC protocol pays the
# configured latency repeatedly over many communication rounds.
RUN_WAN = True
WAN_N = 8
WAN_ONE_WAY_DELAY_MS = 20
WAN_REPEATS = 3
WAN_MIN_WALL_SLOWDOWN = 1.20

PROJECT = Path('/content/sle_journal_technical')
print({
    'profile': RUN_PROFILE,
    'integrated_N': BIOMETRIC_IDENTITIES,
    'root_bits': BIOMETRIC_ROOT_BITS,
    'locator_bits': BIOMETRIC_LOCATOR_BITS,
    'scaling_N': SCALING_N,
    'wan_one_way_ms': WAN_ONE_WAY_DELAY_MS,
})


## Environment and guarded command runner

Package installation does not pin or downgrade NumPy, PyTorch, or
pandas. This avoids the binary-incompatibility error seen in earlier
Colab sessions. MP-SPDZ compilation uses two jobs to control RAM.


In [ ]:
# CELL 2. Imports, dependency repair, and MP-SPDZ build
import importlib.util
import io
import json
import os
import random
import re
import shutil
import subprocess
import sys
import threading
import time
import zipfile
from collections import defaultdict
from pathlib import Path

def install_if_missing(module_name, pip_args):
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', *pip_args
        ])

install_if_missing('facenet_pytorch', ['--no-deps', 'facenet-pytorch==2.6.0'])
install_if_missing('datasets', ['datasets'])
install_if_missing('psutil', ['psutil'])

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import torch
import torch.nn.functional as F
from PIL import Image as PILImage
from datasets import Image as HFImage, load_dataset
from facenet_pytorch import InceptionResnetV1
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from tqdm.auto import tqdm

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT = Path('/content/sle_journal_technical')
PROJECT.mkdir(parents=True, exist_ok=True)
RUNS = PROJECT / 'runs'
RUNS.mkdir(exist_ok=True)
MPSPDZ = Path('/content/MP-SPDZ')

def run(command, cwd=None, env=None, log_path=None, check=True):
    print('$', command)
    started = time.perf_counter()
    process = subprocess.Popen(
        command, cwd=cwd, env=env, shell=True, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    lines = []
    for line in process.stdout:
        print(line, end='')
        lines.append(line)
    returncode = process.wait()
    output = ''.join(lines)
    elapsed = time.perf_counter() - started
    print(f'elapsed={elapsed:.2f}s returncode={returncode}')
    if log_path is not None:
        Path(log_path).write_text(output, encoding='utf-8')
    if check and returncode:
        raise RuntimeError(command)
    return output, elapsed, returncode

def run_monitored(command, cwd=None, env=None, log_path=None):
    print('$', command)
    started = time.perf_counter()
    process = subprocess.Popen(
        command, cwd=cwd, env=env, shell=True, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    lines = []

    def consume():
        for line in process.stdout:
            print(line, end='')
            lines.append(line)

    reader = threading.Thread(target=consume, daemon=True)
    reader.start()
    peak_rss = 0
    while process.poll() is None:
        try:
            parent = psutil.Process(process.pid)
            family = [parent] + parent.children(recursive=True)
            rss = sum(
                item.memory_info().rss for item in family
                if item.is_running()
            )
            peak_rss = max(peak_rss, rss)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
        time.sleep(0.05)

    returncode = process.wait()
    reader.join(timeout=5)
    output = ''.join(lines)
    elapsed = time.perf_counter() - started
    peak_mb = peak_rss / (1024 ** 2)
    print(
        f'elapsed={elapsed:.2f}s returncode={returncode} '
        f'peak_process_tree_rss={peak_mb:.1f} MB'
    )
    if log_path is not None:
        Path(log_path).write_text(output, encoding='utf-8')
    if returncode:
        raise RuntimeError(command)
    return output, elapsed, peak_mb

# Repair interrupted apt/dpkg state before installing build tools.
run('DEBIAN_FRONTEND=noninteractive dpkg --configure -a')
run('DEBIAN_FRONTEND=noninteractive apt-get -qq --fix-broken install -y')
run(
    'apt-get -qq update && DEBIAN_FRONTEND=noninteractive '
    'apt-get -qq install -y automake build-essential clang cmake git '
    'iproute2 iputils-ping libboost-dev libboost-filesystem-dev '
    'libboost-iostreams-dev libboost-thread-dev libgmp-dev libntl-dev '
    'libsodium-dev libssl-dev libtool'
)
if not MPSPDZ.exists():
    run('git clone --depth 1 https://github.com/data61/MP-SPDZ.git /content/MP-SPDZ')

# Pin the framework revision used by the original journal run. This
# removes patches left by earlier executions in the same runtime.
run(f'git fetch --depth 1 origin {MP_SPDZ_REVISION}', cwd=MPSPDZ)
run(f'git checkout --detach --force {MP_SPDZ_REVISION}', cwd=MPSPDZ)

# Detect, but never install, the obsolete plaintext-socket hook from
# an earlier notebook revision. The audited SSL-message hook below is
# the only fallback used by this notebook.
sockets_cpp = MPSPDZ / 'Networking' / 'sockets.cpp'
sockets_text = sockets_cpp.read_text(encoding='utf-8')
delay_marker = 'SLE_EMULATED_SEND_DELAY_US'
if False and delay_marker not in sockets_text:
    if '#include <cstdlib>' not in sockets_text:
        sockets_text = sockets_text.replace(
            '#include <fcntl.h>\n',
            '#include <fcntl.h>\n#include <cstdlib>\n',
            1,
        )
    old_send = 'void send(int socket, octet* msg, size_t len)\n{\n'
    new_send = (
        'void send(int socket, octet* msg, size_t len)\n'
        '{\n'
        '  // Optional journal-benchmark latency injection. Unset in normal runs.\n'
        '  const char* sle_delay = getenv("SLE_EMULATED_SEND_DELAY_US");\n'
        '  if (sle_delay)\n'
        '    {\n'
        '      long delay_us = atol(sle_delay);\n'
        '      if (delay_us > 0)\n'
        '        usleep(delay_us);\n'
        '    }\n'
    )
    if old_send not in sockets_text:
        raise RuntimeError('Could not locate MP-SPDZ send() for WAN hook')
    sockets_cpp.write_text(
        sockets_text.replace(old_send, new_send, 1),
        encoding='utf-8',
    )
USERSPACE_DELAY_PATCHED = delay_marker in sockets_cpp.read_text(
    encoding='utf-8'
)

# The legacy hook above only covers plaintext setup sockets. Install
# the validated fallback at octetStream::Send(), which is the logical
# message path used by SSL-protected MPC traffic as well. Every WAN
# process reports how many logical messages were actually delayed.
octet_header = MPSPDZ / 'Tools' / 'octetStream.h'
octet_text = octet_header.read_text(encoding='utf-8')
transport_delay_marker = 'SLE_WAN_MESSAGE_DELAY_US'
if transport_delay_marker not in octet_text:
    include_anchor = '#include <assert.h>\n'
    include_patch = (
        '#include <assert.h>\n'
        '#include <atomic>\n'
        '#include <cstdlib>\n'
        '#include <unistd.h>\n'
    )
    if include_anchor not in octet_text:
        raise RuntimeError('Could not locate octetStream include anchor')
    octet_text = octet_text.replace(include_anchor, include_patch, 1)

    send_anchor = (
        'template<class T>\n'
        'inline void octetStream::Send(T socket_num) const\n'
        '{\n'
    )
    delay_hook = '\n'.join([
        'inline std::atomic<unsigned long long>& sle_wan_delayed_messages()',
        '{',
        '  static std::atomic<unsigned long long> value(0);',
        '  return value;',
        '}',
        '',
        'inline std::atomic<unsigned long long>& sle_wan_delayed_bytes()',
        '{',
        '  static std::atomic<unsigned long long> value(0);',
        '  return value;',
        '}',
        '',
        'inline long sle_wan_message_delay_us()',
        '{',
        '  static long value = []() {',
        '    const char* raw = std::getenv("SLE_WAN_MESSAGE_DELAY_US");',
        '    return raw ? std::atol(raw) : 0L;',
        '  }();',
        '  return value;',
        '}',
        '',
        'inline void sle_wan_report()',
        '{',
        '  std::fprintf(stderr,',
        '      "SLE_WAN_DELAY_REPORT delay_us=%ld delayed_messages=%llu delayed_bytes=%llu\\n",',
        '      sle_wan_message_delay_us(),',
        '      (unsigned long long) sle_wan_delayed_messages().load(),',
        '      (unsigned long long) sle_wan_delayed_bytes().load());',
        '}',
        '',
        'inline void sle_wan_delay_logical_message(size_t payload_bytes)',
        '{',
        '  const long delay_us = sle_wan_message_delay_us();',
        '  if (delay_us <= 0)',
        '    return;',
        '',
        '  static std::atomic<bool> report_registered(false);',
        '  bool expected = false;',
        '  if (report_registered.compare_exchange_strong(expected, true))',
        '    std::atexit(sle_wan_report);',
        '',
        '  sle_wan_delayed_messages().fetch_add(1);',
        '  sle_wan_delayed_bytes().fetch_add(payload_bytes);',
        '  usleep(delay_us);',
        '}',
        '',
        'template<class T>',
        'inline void octetStream::Send(T socket_num) const',
        '{',
        '  sle_wan_delay_logical_message(get_length());',
    ]) + '\n'
    if send_anchor not in octet_text:
        raise RuntimeError(
            'Could not locate octetStream::Send() for WAN hook'
        )
    octet_header.write_text(
        octet_text.replace(send_anchor, delay_hook, 1),
        encoding='utf-8',
    )
USERSPACE_DELAY_PATCHED = (
    transport_delay_marker in octet_header.read_text(encoding='utf-8')
)
run('make -j2 setup', cwd=MPSPDZ)
run('make -j2 shamir-party.x', cwd=MPSPDZ)
run(f'Scripts/setup-ssl.sh {PLAYERS}', cwd=MPSPDZ)
commit, _, _ = run('git rev-parse HEAD', cwd=MPSPDZ)
MP_SPDZ_COMMIT = commit.strip().splitlines()[-1]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('MP-SPDZ commit:', MP_SPDZ_COMMIT)
print('FaceNet device:', DEVICE)
print('User-space WAN fallback installed:', USERSPACE_DELAY_PATCHED)


## Embedded protocol sources

Only this notebook needs to be uploaded to Colab. The protocol source
and journal-specific input generator are written into the runtime by
the next cells.


In [ ]:
target = PROJECT / 'Programs/Source/sle_integrated_transition_match.mpc'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""\nIntegrated SLE epoch transition and binary biometric identification.\n\nThe circuit consumes one protected database state, its padded candidate index,\nand one binary biometric probe. It then:\n\n  1. evolves every database and index copy of a locator with the same hidden\n     epoch permutation;\n  2. verifies the hidden (locator, block, bucket) multiset relation;\n  3. atomically commits or rolls back the candidate state; and\n  4. performs a complete secret Hamming/ArgMin identification over the same\n     binary roots.\n\nOnly relation_ok, accept/reject, and the final identity are opened. Roots,\nlocators, keys, distances, and the old-to-new relation remain secret-shared.\n\nCompile-time arguments:\n    sle_integrated_transition_match <n_records> <root_bits> <block_bits>\n                                    <bucket_capacity> <epoch>\n                                    <accept_threshold> <locator_bits>\n"""\n\nfrom Compiler.library import tree_reduce\n\nprogram.use_edabit(True)\nprogram.set_bit_length(64)\n\nif len(program.args) != 8:\n    raise CompilerError(\n        "Usage: sle_integrated_transition_match <n_records> <root_bits> "\n        "<block_bits> <bucket_capacity> <epoch> <accept_threshold> "\n        "<locator_bits>"\n    )\n\nN_RECORDS = int(program.args[1])\nROOT_BITS = int(program.args[2])\nBLOCK_BITS = int(program.args[3])\nBUCKET_CAPACITY = int(program.args[4])\nNEW_EPOCH = int(program.args[5])\nACCEPT_THRESHOLD = int(program.args[6])\nLOCATOR_BITS = int(program.args[7])\n\nif N_RECORDS < 2:\n    raise CompilerError("n_records must be at least two")\nif ROOT_BITS < 8 or ROOT_BITS % BLOCK_BITS != 0:\n    raise CompilerError("root_bits must be divisible by block_bits")\nif BLOCK_BITS < 1 or BLOCK_BITS > 8:\n    raise CompilerError("block_bits must be in [1, 8]")\nif BUCKET_CAPACITY < 1:\n    raise CompilerError("bucket_capacity must be positive")\nif not 0 <= ACCEPT_THRESHOLD <= ROOT_BITS:\n    raise CompilerError("accept_threshold must be in [0, root_bits]")\nif LOCATOR_BITS < 16 or LOCATOR_BITS > 64:\n    raise CompilerError("locator_bits must be in [16, 64]")\n\nN_BLOCKS = ROOT_BITS // BLOCK_BITS\nBUCKET_VALUES = 2 ** BLOCK_BITS\nN_BUCKETS = N_BLOCKS * BUCKET_VALUES\nN_INDEX_SLOTS = N_BUCKETS * BUCKET_CAPACITY\nMIMC_ROUNDS = 73\n\n\n# Player 0 owns the enrolled state. All values become secret shares.\nold_db_locators = sint.Array(N_RECORDS)\nroots = sint.Matrix(N_RECORDS, ROOT_BITS)\nold_index_locators = sint.Array(N_INDEX_SLOTS)\nindex_valid = sint.Array(N_INDEX_SLOTS)\nlabels = sint.Array(N_RECORDS)\ntie_tags = sint.Array(N_RECORDS)\n\nold_db_locators.input_from(0)\nroots.input_from(0)\nold_index_locators.input_from(0)\nindex_valid.input_from(0)\nlabels.input_from(0)\ntie_tags.input_from(0)\n\n\n# Every computing party supplies one fresh contribution. The complete epoch\n# key is assembled only inside MPC.\nepoch_key = sint(0)\nfor party in range(5):\n    epoch_key += sint.get_input_from(party)\n\n\n# Player 1 independently supplies the biometric probe.\nprobe = sint.Array(ROOT_BITS)\nprobe.input_from(1)\n\n\nMIMC_CONSTANTS = [\n    int.from_bytes(\n        __import__("hashlib").sha256(\n            ("SLE-MiMC-round-%d" % round_no).encode()\n        ).digest()[:8],\n        "big",\n    )\n    for round_no in range(MIMC_ROUNDS)\n]\n\n\ndef secure_locator_prp_vector(values):\n    """Vectorized keyed field permutation for one epoch transition."""\n\n    x = values + NEW_EPOCH\n    for constant in MIMC_CONSTANTS:\n        x = (x + epoch_key + constant) ** 3\n    return x + epoch_key\n\n\ndef root_bucket_value(root, block):\n    value = sint(0)\n    for bit in range(BLOCK_BITS):\n        value += root[block * BLOCK_BITS + bit] * (2 ** bit)\n    return value\n\n\ndef product_tree(factors):\n    if not factors:\n        return sint(1)\n    return tree_reduce(lambda left, right: left * right, factors)\n\n\ndef secure_hamming(enrolled, query):\n    return sum(\n        enrolled[bit] + query[bit] - 2 * enrolled[bit] * query[bit]\n        for bit in range(ROOT_BITS)\n    )\n\n\nstart_timer(7)\n\n# SLE transition: equal locator copies receive the same hidden permutation.\nnew_db_locators = sint.Array(N_RECORDS)\nnew_index_locators = sint.Array(N_INDEX_SLOTS)\nnew_index_valid = sint.Array(N_INDEX_SLOTS)\n\nnew_db_locators.assign_vector(\n    secure_locator_prp_vector(old_db_locators.get_vector())\n)\nnew_index_locators.assign_vector(\n    secure_locator_prp_vector(old_index_locators.get_vector())\n)\nnew_index_valid.assign_vector(index_valid.get_vector())\n\n\n# Two independently challenged multiset fingerprints verify that every\n# semantic index element is present with the right locator, block, and bucket.\nchallenge_1 = sint.get_random_int(60).reveal() + 101\nchallenge_2 = sint.get_random_int(60).reveal() + 211\nalpha_1 = cint(1_000_003)\nbeta_1 = cint(1_000_033)\nalpha_2 = cint(1_000_037)\nbeta_2 = cint(1_000_081)\n\nexpected_factors_1 = []\nexpected_factors_2 = []\nfor row in range(N_RECORDS):\n    locator = new_db_locators[row]\n    for block in range(N_BLOCKS):\n        bucket = root_bucket_value(roots[row], block)\n        expected_factors_1.append(\n            challenge_1 - (locator + alpha_1 * block + beta_1 * bucket)\n        )\n        expected_factors_2.append(\n            challenge_2 - (locator + alpha_2 * block + beta_2 * bucket)\n        )\n\nactual_factors_1 = []\nactual_factors_2 = []\nfor position in range(N_INDEX_SLOTS):\n    bucket_address = position // BUCKET_CAPACITY\n    block = bucket_address // BUCKET_VALUES\n    bucket = bucket_address % BUCKET_VALUES\n    locator = new_index_locators[position]\n    valid = new_index_valid[position]\n    actual_factors_1.append(\n        valid * (\n            challenge_1 - (locator + alpha_1 * block + beta_1 * bucket)\n        ) + (1 - valid)\n    )\n    actual_factors_2.append(\n        valid * (\n            challenge_2 - (locator + alpha_2 * block + beta_2 * bucket)\n        ) + (1 - valid)\n    )\n\nrelation_ok = (\n    (product_tree(expected_factors_1) == product_tree(actual_factors_1))\n    * (product_tree(expected_factors_2) == product_tree(actual_factors_2))\n)\n\n\n# Atomic state selection. A failed relation check retains the old state.\ncommitted_db_locators = sint.Array(N_RECORDS)\ncommitted_index_locators = sint.Array(N_INDEX_SLOTS)\ncommitted_index_valid = sint.Array(N_INDEX_SLOTS)\n\nfor row in range(N_RECORDS):\n    committed_db_locators[row] = relation_ok.if_else(\n        new_db_locators[row], old_db_locators[row]\n    )\n\nfor position in range(N_INDEX_SLOTS):\n    committed_index_locators[position] = relation_ok.if_else(\n        new_index_locators[position], old_index_locators[position]\n    )\n    committed_index_valid[position] = relation_ok.if_else(\n        new_index_valid[position], index_valid[position]\n    )\n\ncommitted_db_locators.write_to_file()\ncommitted_index_locators.write_to_file()\ncommitted_index_valid.write_to_file()\n\n\n# Closed-set 1:N identification over the same roots. The decision is released\n# only when the transition relation is valid.\nbest_distance = sint(ROOT_BITS + 1)\nbest_label = sint(-1)\nbest_tie_tag = sint(2 ** 31 - 1)\n\nfor row in range(N_RECORDS):\n    distance = secure_hamming(roots[row], probe)\n    strictly_closer = distance < best_distance\n    equal_distance = distance == best_distance\n    smaller_tie_tag = tie_tags[row] < best_tie_tag\n    better = strictly_closer + equal_distance * smaller_tie_tag\n\n    best_distance = better.if_else(distance, best_distance)\n    best_label = better.if_else(labels[row], best_label)\n    best_tie_tag = better.if_else(tie_tags[row], best_tie_tag)\n\naccepted = relation_ok * (best_distance <= ACCEPT_THRESHOLD)\nopened_relation_ok = relation_ok.reveal()\nopened_accept = accepted.reveal()\nopened_decision = accepted.if_else(best_label, sint(-1)).reveal()\n\nstop_timer(7)\n\nprint_ln(\n    "SLE_MPC_RESULT mode=integrated n=%s root_bits=%s locator_bits=%s "\n    "relation_ok=%s accepted=%s identity=%s",\n    N_RECORDS,\n    ROOT_BITS,\n    LOCATOR_BITS,\n    opened_relation_ok,\n    opened_accept,\n    opened_decision,\n)\nprint_ln(\n    "SLE_MPC_PRIVATE roots=not_opened distances=not_opened "\n    "locators=not_opened key=not_opened mapping=not_materialized"\n)\n', encoding='utf-8')
print('Wrote:', target)


In [ ]:
target = PROJECT / 'Programs/Source/sle_transition_optimized.mpc'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""\nRound-optimized MPC implementation of one SLE epoch transition.\n\nThe semantics match sle_transition.mpc:\n  * five parties jointly contribute the epoch key;\n  * database and index copies evolve without opening a locator;\n  * the hidden (locator, block, bucket) relation is checked twice;\n  * only relation_ok is opened and state selection is atomic.\n\nThe implementation changes two performance details only. First, all locator\ncopies are evaluated as register vectors, so independent PRP instances are\nbatched across the same 73 sequential rounds. Second, multiset fingerprints\nuse a balanced product tree instead of one long dependency chain.\n\nCompile-time arguments:\n    sle_transition_optimized <n_records> <root_bits> <block_bits>\n                             <bucket_capacity> <epoch> <fault_mode>\n"""\n\nfrom Compiler.library import tree_reduce\n\nprogram.use_edabit(True)\nprogram.set_bit_length(64)\n\nif len(program.args) != 7:\n    raise CompilerError(\n        "Usage: sle_transition_optimized <n_records> <root_bits> "\n        "<block_bits> <bucket_capacity> <epoch> <fault_mode>"\n    )\n\nN_RECORDS = int(program.args[1])\nROOT_BITS = int(program.args[2])\nBLOCK_BITS = int(program.args[3])\nBUCKET_CAPACITY = int(program.args[4])\nNEW_EPOCH = int(program.args[5])\nFAULT_MODE = int(program.args[6])\n\nif N_RECORDS < 2:\n    raise CompilerError("n_records must be at least two")\nif ROOT_BITS % BLOCK_BITS != 0:\n    raise CompilerError("root_bits must be divisible by block_bits")\nif FAULT_MODE not in (0, 1, 2, 3):\n    raise CompilerError("fault_mode must be 0, 1, 2, or 3")\n\nN_BLOCKS = ROOT_BITS // BLOCK_BITS\nBUCKET_VALUES = 2 ** BLOCK_BITS\nN_BUCKETS = N_BLOCKS * BUCKET_VALUES\nN_INDEX_SLOTS = N_BUCKETS * BUCKET_CAPACITY\nMIMC_ROUNDS = 73\n\n\nold_db_locators = sint.Array(N_RECORDS)\nroots = sint.Matrix(N_RECORDS, ROOT_BITS)\nold_index_locators = sint.Array(N_INDEX_SLOTS)\nindex_valid = sint.Array(N_INDEX_SLOTS)\n\nold_db_locators.input_from(0)\nroots.input_from(0)\nold_index_locators.input_from(0)\nindex_valid.input_from(0)\n\n# The complete epoch key is never supplied by one party.\nepoch_key = sint(0)\nfor party in range(5):\n    epoch_key += sint.get_input_from(party)\n\nMIMC_CONSTANTS = [\n    int.from_bytes(\n        __import__("hashlib").sha256(\n            ("SLE-MiMC-round-%d" % round_no).encode()\n        ).digest()[:8],\n        "big",\n    )\n    for round_no in range(MIMC_ROUNDS)\n]\n\n\ndef secure_locator_prp_vector(values):\n    """Evaluate independent locator transitions in one vectorized circuit."""\n\n    x = values + NEW_EPOCH\n    for constant in MIMC_CONSTANTS:\n        x = (x + epoch_key + constant) ** 3\n    return x + epoch_key\n\n\ndef root_bucket_value(root, block):\n    value = sint(0)\n    for bit in range(BLOCK_BITS):\n        value += root[block * BLOCK_BITS + bit] * (2 ** bit)\n    return value\n\n\ndef product_tree(factors):\n    """Multiply factors with logarithmic dependency depth."""\n\n    if not factors:\n        return sint(1)\n    return tree_reduce(lambda left, right: left * right, factors)\n\n\nstart_timer(3)\n\nnew_db_locators = sint.Array(N_RECORDS)\nnew_index_locators = sint.Array(N_INDEX_SLOTS)\nnew_index_valid = sint.Array(N_INDEX_SLOTS)\n\nnew_db_locators.assign_vector(\n    secure_locator_prp_vector(old_db_locators.get_vector())\n)\nnew_index_locators.assign_vector(\n    secure_locator_prp_vector(old_index_locators.get_vector())\n)\nnew_index_valid.assign_vector(index_valid.get_vector())\n\n# These controlled faults test the relation gate. They are public test modes,\n# not a claim of active security for the passive Shamir backend.\nif FAULT_MODE == 1:\n    new_index_valid[0] = sint(0)\nelif FAULT_MODE == 2:\n    new_index_locators[0] = new_index_locators[0] + 1\nelif FAULT_MODE == 3:\n    temporary = new_index_locators[0]\n    new_index_locators[0] = new_index_locators[BUCKET_CAPACITY]\n    new_index_locators[BUCKET_CAPACITY] = temporary\n\n\n# Two public challenges are sampled after candidate values have been formed.\nchallenge_1 = sint.get_random_int(60).reveal() + 101\nchallenge_2 = sint.get_random_int(60).reveal() + 211\nalpha_1 = cint(1_000_003)\nbeta_1 = cint(1_000_033)\nalpha_2 = cint(1_000_037)\nbeta_2 = cint(1_000_081)\n\nexpected_factors_1 = []\nexpected_factors_2 = []\nfor row in range(N_RECORDS):\n    locator = new_db_locators[row]\n    for block in range(N_BLOCKS):\n        bucket = root_bucket_value(roots[row], block)\n        element_1 = locator + alpha_1 * block + beta_1 * bucket\n        element_2 = locator + alpha_2 * block + beta_2 * bucket\n        expected_factors_1.append(challenge_1 - element_1)\n        expected_factors_2.append(challenge_2 - element_2)\n\nactual_factors_1 = []\nactual_factors_2 = []\nfor position in range(N_INDEX_SLOTS):\n    bucket_address = position // BUCKET_CAPACITY\n    block = bucket_address // BUCKET_VALUES\n    bucket = bucket_address % BUCKET_VALUES\n    locator = new_index_locators[position]\n    valid = new_index_valid[position]\n\n    element_1 = locator + alpha_1 * block + beta_1 * bucket\n    element_2 = locator + alpha_2 * block + beta_2 * bucket\n    actual_factors_1.append(\n        valid * (challenge_1 - element_1) + (1 - valid)\n    )\n    actual_factors_2.append(\n        valid * (challenge_2 - element_2) + (1 - valid)\n    )\n\nexpected_1 = product_tree(expected_factors_1)\nexpected_2 = product_tree(expected_factors_2)\nactual_1 = product_tree(actual_factors_1)\nactual_2 = product_tree(actual_factors_2)\nrelation_ok = (expected_1 == actual_1) * (expected_2 == actual_2)\n\n\n# A failed transition preserves the old shared state.\ncommitted_db_locators = sint.Array(N_RECORDS)\ncommitted_index_locators = sint.Array(N_INDEX_SLOTS)\ncommitted_index_valid = sint.Array(N_INDEX_SLOTS)\n\nfor row in range(N_RECORDS):\n    committed_db_locators[row] = relation_ok.if_else(\n        new_db_locators[row], old_db_locators[row]\n    )\n\nfor position in range(N_INDEX_SLOTS):\n    committed_index_locators[position] = relation_ok.if_else(\n        new_index_locators[position], old_index_locators[position]\n    )\n    committed_index_valid[position] = relation_ok.if_else(\n        new_index_valid[position], index_valid[position]\n    )\n\ncommitted_db_locators.write_to_file()\ncommitted_index_locators.write_to_file()\ncommitted_index_valid.write_to_file()\n\nopened_relation_ok = relation_ok.reveal()\nstop_timer(3)\n\nprint_ln(\n    "SLE_MPC_RESULT mode=transition_optimized epoch=%s relation_ok=%s fault=%s",\n    NEW_EPOCH,\n    opened_relation_ok,\n    FAULT_MODE,\n)\nprint_ln(\n    "SLE_MPC_PRIVATE key=not_opened locators=not_opened "\n    "mapping=not_materialized"\n)\n', encoding='utf-8')
print('Wrote:', target)


In [ ]:
target = PROJECT / 'Programs/Source/sle_pointer_join_transition.mpc'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""\nGeneric secret pointer-translation baseline for an SLE epoch transition.\n\nThe baseline gives SLE a meaningful competitor under the same five-party\nShamir execution and the same opened output. It first evolves the N database\nlocators. Every one of the M padded index entries is then joined against all N\nold database locators with secret equality tests in order to recover the\ncorresponding new locator. No pointer map is opened, but O(N*M) secure equality\nwork is required.\n\nThis is deliberately not presented as a novel protocol. It is an executable\ngeneric MPC baseline for the pointer-translation step that SLE avoids.\n\nCompile-time arguments:\n    sle_pointer_join_transition <n_records> <root_bits> <block_bits>\n                                <bucket_capacity> <epoch>\n"""\n\nfrom Compiler.library import tree_reduce\n\nprogram.use_edabit(True)\nprogram.set_bit_length(64)\n\nif len(program.args) != 6:\n    raise CompilerError(\n        "Usage: sle_pointer_join_transition <n_records> <root_bits> "\n        "<block_bits> <bucket_capacity> <epoch>"\n    )\n\nN_RECORDS = int(program.args[1])\nROOT_BITS = int(program.args[2])\nBLOCK_BITS = int(program.args[3])\nBUCKET_CAPACITY = int(program.args[4])\nNEW_EPOCH = int(program.args[5])\n\nif N_RECORDS < 2:\n    raise CompilerError("n_records must be at least two")\nif ROOT_BITS % BLOCK_BITS != 0:\n    raise CompilerError("root_bits must be divisible by block_bits")\n\nN_BLOCKS = ROOT_BITS // BLOCK_BITS\nBUCKET_VALUES = 2 ** BLOCK_BITS\nN_BUCKETS = N_BLOCKS * BUCKET_VALUES\nN_INDEX_SLOTS = N_BUCKETS * BUCKET_CAPACITY\nMIMC_ROUNDS = 73\n\n\nold_db_locators = sint.Array(N_RECORDS)\nroots = sint.Matrix(N_RECORDS, ROOT_BITS)\nold_index_locators = sint.Array(N_INDEX_SLOTS)\nindex_valid = sint.Array(N_INDEX_SLOTS)\n\nold_db_locators.input_from(0)\nroots.input_from(0)\nold_index_locators.input_from(0)\nindex_valid.input_from(0)\n\nepoch_key = sint(0)\nfor party in range(5):\n    epoch_key += sint.get_input_from(party)\n\nMIMC_CONSTANTS = [\n    int.from_bytes(\n        __import__("hashlib").sha256(\n            ("SLE-MiMC-round-%d" % round_no).encode()\n        ).digest()[:8],\n        "big",\n    )\n    for round_no in range(MIMC_ROUNDS)\n]\n\n\ndef secure_locator_prp_vector(values):\n    x = values + NEW_EPOCH\n    for constant in MIMC_CONSTANTS:\n        x = (x + epoch_key + constant) ** 3\n    return x + epoch_key\n\n\ndef root_bucket_value(root, block):\n    value = sint(0)\n    for bit in range(BLOCK_BITS):\n        value += root[block * BLOCK_BITS + bit] * (2 ** bit)\n    return value\n\n\ndef product_tree(factors):\n    if not factors:\n        return sint(1)\n    return tree_reduce(lambda left, right: left * right, factors)\n\n\nstart_timer(4)\n\n# Database transition is identical to optimized SLE.\nnew_db_locators = sint.Array(N_RECORDS)\nnew_db_locators.assign_vector(\n    secure_locator_prp_vector(old_db_locators.get_vector())\n)\n\n# Generic pointer translation. Each valid index copy must discover which old\n# database locator it refers to before selecting the corresponding new value.\nnew_index_locators = sint.Array(N_INDEX_SLOTS)\nnew_index_valid = sint.Array(N_INDEX_SLOTS)\njoin_checks = []\n\nfor position in range(N_INDEX_SLOTS):\n    valid = index_valid[position]\n    translated = sint(0)\n    number_of_matches = sint(0)\n\n    for row in range(N_RECORDS):\n        is_match = old_index_locators[position] == old_db_locators[row]\n        translated += is_match * new_db_locators[row]\n        number_of_matches += is_match\n\n    # Dummy slots receive fresh secret values. For valid entries the equality\n    # join must find exactly one database record; dummies must find none.\n    dummy_locator = sint.get_random_int(60)\n    new_index_locators[position] = valid.if_else(\n        translated, dummy_locator\n    )\n    new_index_valid[position] = valid\n    join_checks.append(\n        valid * (number_of_matches == 1)\n        + (1 - valid) * (number_of_matches == 0)\n    )\n\njoin_ok = product_tree(join_checks)\n\n# Use the same hidden relation check as SLE so both transitions have the same\n# integrity output and commit semantics.\nchallenge_1 = sint.get_random_int(60).reveal() + 101\nchallenge_2 = sint.get_random_int(60).reveal() + 211\nalpha_1 = cint(1_000_003)\nbeta_1 = cint(1_000_033)\nalpha_2 = cint(1_000_037)\nbeta_2 = cint(1_000_081)\n\nexpected_factors_1 = []\nexpected_factors_2 = []\nfor row in range(N_RECORDS):\n    locator = new_db_locators[row]\n    for block in range(N_BLOCKS):\n        bucket = root_bucket_value(roots[row], block)\n        expected_factors_1.append(\n            challenge_1 - (locator + alpha_1 * block + beta_1 * bucket)\n        )\n        expected_factors_2.append(\n            challenge_2 - (locator + alpha_2 * block + beta_2 * bucket)\n        )\n\nactual_factors_1 = []\nactual_factors_2 = []\nfor position in range(N_INDEX_SLOTS):\n    bucket_address = position // BUCKET_CAPACITY\n    block = bucket_address // BUCKET_VALUES\n    bucket = bucket_address % BUCKET_VALUES\n    locator = new_index_locators[position]\n    valid = new_index_valid[position]\n    actual_factors_1.append(\n        valid * (\n            challenge_1 - (locator + alpha_1 * block + beta_1 * bucket)\n        ) + (1 - valid)\n    )\n    actual_factors_2.append(\n        valid * (\n            challenge_2 - (locator + alpha_2 * block + beta_2 * bucket)\n        ) + (1 - valid)\n    )\n\nrelation_ok = (\n    join_ok\n    * (product_tree(expected_factors_1) == product_tree(actual_factors_1))\n    * (product_tree(expected_factors_2) == product_tree(actual_factors_2))\n)\n\ncommitted_db_locators = sint.Array(N_RECORDS)\ncommitted_index_locators = sint.Array(N_INDEX_SLOTS)\ncommitted_index_valid = sint.Array(N_INDEX_SLOTS)\n\nfor row in range(N_RECORDS):\n    committed_db_locators[row] = relation_ok.if_else(\n        new_db_locators[row], old_db_locators[row]\n    )\n\nfor position in range(N_INDEX_SLOTS):\n    committed_index_locators[position] = relation_ok.if_else(\n        new_index_locators[position], old_index_locators[position]\n    )\n    committed_index_valid[position] = relation_ok.if_else(\n        new_index_valid[position], index_valid[position]\n    )\n\ncommitted_db_locators.write_to_file()\ncommitted_index_locators.write_to_file()\ncommitted_index_valid.write_to_file()\n\nopened_relation_ok = relation_ok.reveal()\nstop_timer(4)\n\nprint_ln(\n    "SLE_MPC_RESULT mode=pointer_join epoch=%s relation_ok=%s",\n    NEW_EPOCH,\n    opened_relation_ok,\n)\nprint_ln(\n    "SLE_MPC_PRIVATE key=not_opened locators=not_opened "\n    "pointer_map=not_opened"\n)\n', encoding='utf-8')
print('Wrote:', target)


In [ ]:
target = PROJECT / 'scripts/generate_journal_inputs.py'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('#!/usr/bin/env python3\n"""Generate reproducible inputs for journal-scale SLE experiments.\n\nUnlike the original smoke-test generator, biometric-root width and locator\nwidth are independent parameters. Locators are Python integers, so a 64-bit\nstorage width is exercised without NumPy signed-integer truncation.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\n\nSEED = 20260804\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\n        "--mode", choices=("transition", "integrated"), required=True\n    )\n    parser.add_argument("--output-dir", type=Path, required=True)\n    parser.add_argument("--n-records", type=int, required=True)\n    parser.add_argument("--root-bits", type=int, required=True)\n    parser.add_argument("--block-bits", type=int, default=4)\n    parser.add_argument("--bucket-capacity", type=int, required=True)\n    parser.add_argument("--locator-bits", type=int, default=32)\n    parser.add_argument("--query-row", type=int, default=0)\n    parser.add_argument("--npz", type=Path)\n    parser.add_argument("--flip-probability", type=float, default=0.05)\n    return parser.parse_args()\n\n\ndef flatten(*arrays) -> list[int]:\n    output: list[int] = []\n    for array in arrays:\n        if isinstance(array, np.ndarray):\n            output.extend(int(value) for value in array.reshape(-1))\n        elif isinstance(array, (list, tuple)):\n            output.extend(int(value) for value in array)\n        else:\n            output.append(int(array))\n    return output\n\n\ndef load_roots(args: argparse.Namespace):\n    rng = np.random.default_rng(SEED)\n\n    if args.npz is None:\n        gallery = rng.integers(\n            0, 2, size=(args.n_records, args.root_bits), dtype=np.uint8\n        )\n        labels = np.arange(1000, 1000 + args.n_records, dtype=np.int64)\n        probes = gallery.copy()\n        flips = rng.random(probes.shape) < args.flip_probability\n        probes ^= flips.astype(np.uint8)\n    else:\n        archive = np.load(args.npz)\n        gallery = np.asarray(archive["gallery_roots"], dtype=np.uint8)\n        probes = np.asarray(archive["probe_roots"], dtype=np.uint8)\n        labels = np.asarray(\n            archive.get("labels", archive.get("gallery_labels")),\n            dtype=np.int64,\n        )\n        if gallery.ndim != 2 or probes.ndim != 2:\n            raise ValueError("gallery_roots and probe_roots must be matrices")\n        if len(gallery) < args.n_records:\n            raise ValueError("NPZ has fewer enrolled identities than requested")\n        if gallery.shape[1] < args.root_bits or probes.shape[1] < args.root_bits:\n            raise ValueError("NPZ roots are narrower than --root-bits")\n        gallery = gallery[: args.n_records, : args.root_bits]\n        labels = labels[: args.n_records]\n        probes = probes[:, : args.root_bits]\n\n    if not np.isin(gallery, (0, 1)).all():\n        raise ValueError("gallery roots must be binary")\n    if not np.isin(probes, (0, 1)).all():\n        raise ValueError("probe roots must be binary")\n    if not 0 <= args.query_row < len(probes):\n        raise ValueError("query row is outside probe_roots")\n\n    tie_tags = rng.permutation(\n        np.arange(1, args.n_records + 1, dtype=np.int64)\n    )\n    probe = probes[args.query_row]\n    distances = np.count_nonzero(gallery != probe[None, :], axis=1)\n    best = np.flatnonzero(distances == distances.min())\n    winner = best[np.argmin(tie_tags[best])]\n\n    return gallery, labels, tie_tags, probe, int(labels[winner]), int(distances[winner])\n\n\ndef block_value(root: np.ndarray, block: int, block_bits: int) -> int:\n    start = block * block_bits\n    value = 0\n    for bit, item in enumerate(root[start : start + block_bits]):\n        value |= int(item) << bit\n    return value\n\n\ndef build_bucket_rows(gallery: np.ndarray, block_bits: int, capacity: int):\n    n_records, root_bits = gallery.shape\n    if root_bits % block_bits:\n        raise ValueError("root_bits must be divisible by block_bits")\n\n    n_blocks = root_bits // block_bits\n    bucket_values = 2**block_bits\n    n_buckets = n_blocks * bucket_values\n    rows = np.full((n_buckets, capacity), n_records, dtype=np.int64)\n    valid = np.zeros((n_buckets, capacity), dtype=np.uint8)\n    occupancy = np.zeros(n_buckets, dtype=np.int64)\n\n    for row in range(n_records):\n        for block in range(n_blocks):\n            value = block_value(gallery[row], block, block_bits)\n            address = block * bucket_values + value\n            slot = int(occupancy[address])\n            if slot >= capacity:\n                raise ValueError(\n                    f"bucket {address} needs {slot + 1} slots; "\n                    "increase --bucket-capacity"\n                )\n            rows[address, slot] = row\n            valid[address, slot] = 1\n            occupancy[address] += 1\n\n    return rows, valid, int(occupancy.max())\n\n\ndef random_unique_values(count: int, bits: int, rng: np.random.Generator):\n    if not 16 <= bits <= 64:\n        raise ValueError("locator bits must be in [16, 64]")\n    mask = (1 << bits) - 1\n    values: set[int] = set()\n    while len(values) < count:\n        value = int(rng.bit_generator.random_raw()) & mask\n        if value:\n            values.add(value)\n    return sorted(values)\n\n\ndef build_locators(rows, valid, n_records, locator_bits):\n    rng = np.random.default_rng(SEED + locator_bits + n_records)\n    db_locators = random_unique_values(n_records, locator_bits, rng)\n    forbidden = set(db_locators)\n\n    index_locators = np.empty(rows.shape, dtype=object)\n    for address in range(rows.shape[0]):\n        for slot in range(rows.shape[1]):\n            if valid[address, slot]:\n                index_locators[address, slot] = db_locators[int(rows[address, slot])]\n            else:\n                while True:\n                    candidate = random_unique_values(1, locator_bits, rng)[0]\n                    if candidate not in forbidden:\n                        forbidden.add(candidate)\n                        index_locators[address, slot] = candidate\n                        break\n\n    contributions = [\n        int(value)\n        for value in rng.integers(1, 2**30, size=5, dtype=np.int64)\n    ]\n    return db_locators, index_locators, contributions\n\n\ndef write_inputs(output_dir: Path, values_by_player: dict[int, list[int]]):\n    output_dir.mkdir(parents=True, exist_ok=True)\n    for player in range(5):\n        path = output_dir / f"Input-P{player}-0"\n        values = values_by_player.get(player, [])\n        path.write_text(" ".join(str(int(value)) for value in values) + "\\n")\n\n\ndef sha256_array(array: np.ndarray) -> str:\n    return hashlib.sha256(np.ascontiguousarray(array).tobytes()).hexdigest()\n\n\ndef main() -> None:\n    args = parse_args()\n    if args.n_records < 2:\n        raise ValueError("n-records must be at least two")\n    if args.root_bits % args.block_bits:\n        raise ValueError("root-bits must be divisible by block-bits")\n\n    gallery, labels, tie_tags, probe, expected_identity, expected_distance = load_roots(args)\n    rows, valid, required_capacity = build_bucket_rows(\n        gallery, args.block_bits, args.bucket_capacity\n    )\n    db_locators, index_locators, contributions = build_locators(\n        rows, valid, args.n_records, args.locator_bits\n    )\n\n    player_values = {\n        0: flatten(\n            db_locators,\n            gallery,\n            index_locators,\n            valid,\n        )\n    }\n\n    if args.mode == "integrated":\n        player_values[0].extend(flatten(labels, tie_tags, contributions[0]))\n        player_values[1] = flatten(contributions[1], probe)\n    else:\n        # Trailing row references are consumed by the ORAM-reindex baseline\n        # and ignored by the SLE, pointer-join, and sort/merge programs.\n        player_values[0].extend(flatten(contributions[0], rows))\n        player_values[1] = [contributions[1]]\n\n    for player in range(2, 5):\n        player_values[player] = [contributions[player]]\n\n    write_inputs(args.output_dir, player_values)\n\n    metadata = {\n        "mode": args.mode,\n        "n_records": args.n_records,\n        "root_bits": args.root_bits,\n        "block_bits": args.block_bits,\n        "bucket_capacity": args.bucket_capacity,\n        "required_capacity": required_capacity,\n        "locator_bits": args.locator_bits,\n        "query_row": args.query_row,\n        "expected_identity": expected_identity,\n        "expected_distance": expected_distance,\n        "gallery_sha256": sha256_array(gallery),\n        "probe_sha256": sha256_array(probe),\n        "index_slots": int(rows.size),\n        "semantic_index_entries": int(valid.sum()),\n    }\n    metadata_path = args.output_dir / "sle_journal_input_metadata.json"\n    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")\n    print(json.dumps(metadata, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('Wrote:', target)


In [ ]:
# CELL 7. Install embedded programs into MP-SPDZ
PROGRAM_FILES = [
    'sle_integrated_transition_match.mpc',
    'sle_transition_optimized.mpc',
    'sle_pointer_join_transition.mpc',
]
for name in PROGRAM_FILES:
    shutil.copy2(
        PROJECT / 'Programs/Source' / name,
        MPSPDZ / 'Programs/Source' / name,
    )

INPUTS = MPSPDZ / 'Player-Data'
ENV = os.environ.copy()
ENV.update(PLAYERS=str(PLAYERS), THRESHOLD=str(THRESHOLD))
print('Installed:', PROGRAM_FILES)


## Point 2: 128 identities and 512-bit real biometric roots

CelebA is streamed instead of reconstructed as a multi-gigabyte local
archive. Four photographs are retained per identity: two for gallery
enrollment and two for probes. FaceNet/VGGFace2 produces 512-D vectors;
their signs form 512-bit roots. The floating-point embedding matrix is
deleted after root construction.


In [ ]:
# CELL 8. Stream a deterministic CelebA identity subset
CELEBA_DATASET = 'flwrlabs/celeba'
CELEBA_SPLIT = 'train'
IMAGE_KEY = 'image'
IDENTITY_KEY = 'celeb_id'
MAX_STREAM_ROWS = 202_599

def image_record_to_bytes(record):
    if isinstance(record, dict):
        if record.get('bytes') is not None:
            return bytes(record['bytes'])
        if record.get('path'):
            return Path(record['path']).read_bytes()
    raise ValueError('Unsupported streamed image record')

def decode_rgb(value):
    with PILImage.open(io.BytesIO(value)) as image:
        return image.convert('RGB').copy()

def celeba_stream():
    stream = load_dataset(
        CELEBA_DATASET, split=CELEBA_SPLIT, streaming=True
    )
    return stream.cast_column(IMAGE_KEY, HFImage(decode=False))

def collect_celeba_subset():
    # Pass 1 retains counters only. Holding JPEG bytes for every
    # identity encountered before 128 complete subjects are found can
    # otherwise consume several gigabytes of RAM.
    counts = defaultdict(int)
    complete = set()
    inspected = 0
    for inspected, row in enumerate(celeba_stream(), start=1):
        identity = int(row[IDENTITY_KEY])
        if counts[identity] < IMAGES_PER_ID:
            counts[identity] += 1
            if counts[identity] == IMAGES_PER_ID:
                complete.add(identity)
        if inspected % 5000 == 0:
            print(
                f'pass 1: inspected={inspected:,}; complete identities='
                f'{len(complete)}/{BIOMETRIC_IDENTITIES}'
            )
        if len(complete) >= BIOMETRIC_IDENTITIES:
            break
        if inspected >= MAX_STREAM_ROWS:
            break

    if len(complete) < BIOMETRIC_IDENTITIES:
        raise RuntimeError(
            f'Only {len(complete)} complete identities were found. '
            'Increase MAX_STREAM_ROWS or reduce IMAGES_PER_ID.'
        )

    selected = sorted(complete)[:BIOMETRIC_IDENTITIES]
    selected_set = set(selected)
    collected = {identity: [] for identity in selected}
    remaining = set(selected)

    # Pass 2 retains bytes only for the selected identities.
    for inspected_second, row in enumerate(celeba_stream(), start=1):
        identity = int(row[IDENTITY_KEY])
        if identity in selected_set and len(collected[identity]) < IMAGES_PER_ID:
            collected[identity].append(
                image_record_to_bytes(row[IMAGE_KEY])
            )
            if len(collected[identity]) == IMAGES_PER_ID:
                remaining.discard(identity)
        if inspected_second % 5000 == 0:
            print(
                f'pass 2: inspected={inspected_second:,}; '
                f'completed={BIOMETRIC_IDENTITIES - len(remaining)}/'
                f'{BIOMETRIC_IDENTITIES}'
            )
        if not remaining:
            break
        if inspected_second >= inspected:
            break

    if remaining:
        raise RuntimeError(
            f'Pass 2 could not collect all selected identities: '
            f'{sorted(remaining)[:5]}'
        )
    return {
        identity: [decode_rgb(item) for item in collected[identity]]
        for identity in selected
    }

def synthetic_images():
    rng = np.random.default_rng(SEED)
    result = {}
    for identity in range(BIOMETRIC_IDENTITIES):
        base = rng.integers(30, 225, size=(1, 1, 3), dtype=np.uint8)
        result[identity] = []
        for _ in range(IMAGES_PER_ID):
            noise = rng.normal(0, 18, size=(160, 160, 3))
            array = np.clip(base + noise, 0, 255).astype(np.uint8)
            result[identity].append(PILImage.fromarray(array, mode='RGB'))
    return result

if BIOMETRIC_NPZ is None:
    images_by_identity = (
        synthetic_images()
        if USE_SYNTHETIC_BIOMETRIC_SMOKE
        else collect_celeba_subset()
    )
    selected_ids = sorted(images_by_identity)
    print('Selected identities:', len(selected_ids))
else:
    print('Using precomputed roots:', BIOMETRIC_NPZ)


In [ ]:
# CELL 9. FaceNet extraction, 512-bit roots, and closed-set evaluation
ROOTS_NPZ = PROJECT / 'celeba_128x512_roots.npz'

if BIOMETRIC_NPZ is None:
    transform = transforms.Compose([
        transforms.Resize((160, 160)),
        transforms.ToTensor(),
        transforms.Normalize([0.5] * 3, [0.5] * 3),
    ])

    image_tensors, image_labels, image_roles = [], [], []
    for identity in selected_ids:
        for position, image in enumerate(images_by_identity[identity]):
            image_tensors.append(transform(image))
            image_labels.append(identity)
            image_roles.append(
                'gallery' if position < GALLERY_IMAGES_PER_ID else 'probe'
            )

    tensor = torch.stack(image_tensors)
    labels_array = np.asarray(image_labels, dtype=np.int64)
    roles_array = np.asarray(image_roles)
    loader = DataLoader(
        TensorDataset(tensor), batch_size=64, shuffle=False
    )

    model = InceptionResnetV1(pretrained='vggface2').eval().to(DEVICE)
    batches = []
    with torch.inference_mode():
        for (batch,) in tqdm(loader, desc='FaceNet embeddings'):
            batches.append(F.normalize(model(batch.to(DEVICE)), dim=1).cpu())
    embeddings = torch.cat(batches).numpy().astype(np.float32)

    gallery_roots, gallery_labels = [], []
    probe_roots, probe_labels = [], []
    for identity in selected_ids:
        gallery_mask = (
            (labels_array == identity) & (roles_array == 'gallery')
        )
        probe_mask = (
            (labels_array == identity) & (roles_array == 'probe')
        )
        mean_enrollment = embeddings[gallery_mask].mean(axis=0)
        gallery_roots.append((mean_enrollment >= 0).astype(np.uint8))
        gallery_labels.append(identity)
        probe_roots.extend((embeddings[probe_mask] >= 0).astype(np.uint8))
        probe_labels.extend([identity] * int(probe_mask.sum()))

    gallery_roots = np.asarray(gallery_roots, dtype=np.uint8)
    gallery_labels = np.asarray(gallery_labels, dtype=np.int64)
    probe_roots = np.asarray(probe_roots, dtype=np.uint8)
    probe_labels = np.asarray(probe_labels, dtype=np.int64)
    np.savez_compressed(
        ROOTS_NPZ,
        gallery_roots=gallery_roots,
        probe_roots=probe_roots,
        labels=gallery_labels,
        probe_labels=probe_labels,
    )

    # Delete real-valued embeddings after binary-root construction.
    del embeddings, batches, tensor, image_tensors, model, images_by_identity
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    ROOTS_NPZ = Path(BIOMETRIC_NPZ)
    archive = np.load(ROOTS_NPZ)
    gallery_roots = np.asarray(archive['gallery_roots'], dtype=np.uint8)
    probe_roots = np.asarray(archive['probe_roots'], dtype=np.uint8)
    gallery_labels = np.asarray(archive['labels'], dtype=np.int64)
    probe_labels = np.asarray(archive['probe_labels'], dtype=np.int64)

if gallery_roots.shape != (BIOMETRIC_IDENTITIES, BIOMETRIC_ROOT_BITS):
    raise ValueError(
        f'Expected gallery shape {(BIOMETRIC_IDENTITIES, BIOMETRIC_ROOT_BITS)}, '
        f'got {gallery_roots.shape}'
    )

distances = np.count_nonzero(
    probe_roots[:, None, :] != gallery_roots[None, :, :], axis=2
)
predictions = gallery_labels[np.argmin(distances, axis=1)]
plaintext_rank1 = float(np.mean(predictions == probe_labels))

print('Gallery roots:', gallery_roots.shape)
print('Probe roots:', probe_roots.shape)
print(f'Plain 512-bit closed-set Rank-1: {plaintext_rank1:.4f}')
print('Floating-point embeddings retained: no')

biometric_setup = pd.DataFrame([{
    'dataset': 'synthetic smoke' if USE_SYNTHETIC_BIOMETRIC_SMOKE else 'CelebA subset',
    'identities': BIOMETRIC_IDENTITIES,
    'gallery_roots': len(gallery_roots),
    'probe_roots': len(probe_roots),
    'root_bits': BIOMETRIC_ROOT_BITS,
    'locator_bits': BIOMETRIC_LOCATOR_BITS,
    'plaintext_rank1': plaintext_rank1,
    'embedding_storage': 'deleted after binarization',
}])
biometric_setup.to_csv(PROJECT / 'biometric_setup.csv', index=False)
display(biometric_setup)


In [ ]:
# CELL 10. Shared helpers for capacity, compilation, and log parsing
def required_capacity(gallery, block_bits):
    n, d = gallery.shape
    values = 2 ** block_bits
    occupancy = np.zeros((d // block_bits, values), dtype=np.int64)
    for root in gallery:
        for block in range(d // block_bits):
            value = sum(
                int(root[block * block_bits + bit]) << bit
                for bit in range(block_bits)
            )
            occupancy[block, value] += 1
    return int(occupancy.max())

def synthetic_gallery(n, bits):
    rng = np.random.default_rng(SEED)
    return rng.integers(0, 2, size=(n, bits), dtype=np.uint8)

def compile_program(source_name, arguments, tag):
    log = RUNS / f'{tag}_compile.log'
    text, seconds, _ = run(
        f'python3 compile.py -M {source_name} {arguments}',
        cwd=MPSPDZ, log_path=log,
    )
    program_name = source_name + '-' + arguments.replace(' ', '-')
    return program_name, text, seconds

def last_float(pattern, text):
    values = re.findall(pattern, text)
    return float(values[-1]) if values else np.nan

def last_int(pattern, text):
    values = re.findall(pattern, text)
    return int(values[-1]) if values else np.nan

def parse_measurement(mode, cfg, repeat, compile_text, run_text,
                      compile_seconds, wall_seconds, peak_rss_mb,
                      network='local'):
    timer_values = [
        float(value)
        for value in re.findall(r'Time[0-9]+ = ([0-9.eE+-]+)', run_text)
    ]
    delay_reports = re.findall(
        r'SLE_WAN_DELAY_REPORT delay_us=(\d+) '
        r'delayed_messages=(\d+) delayed_bytes=(\d+)',
        run_text,
    )
    return dict(
        mode=mode,
        network=network,
        repeat=repeat,
        n=cfg['n'],
        root_bits=cfg['root_bits'],
        locator_bits=cfg['locator_bits'],
        block_bits=cfg['block_bits'],
        capacity=cfg['capacity'],
        padded_slots=cfg['padded_slots'],
        passed=('relation_ok=1' in run_text),
        secure_seconds=max(timer_values) if timer_values else np.nan,
        total_seconds=last_float(r'Time = ([0-9.eE+-]+)', run_text),
        wall_seconds=wall_seconds,
        global_mb=last_float(r'Global data sent = ([0-9.eE+-]+) MB', run_text),
        runtime_rounds=last_int(
            r'Data sent = [^\n]*~(\d+) rounds', run_text
        ),
        compiler_vm_rounds=last_int(
            r'(\d+) virtual machine rounds', compile_text
        ),
        integer_triples=last_int(r'(\d+) integer triples', compile_text),
        compile_seconds=compile_seconds,
        peak_rss_mb=peak_rss_mb,
        delay_report_count=len(delay_reports),
        delayed_messages=sum(int(item[1]) for item in delay_reports),
        delayed_bytes=sum(int(item[2]) for item in delay_reports),
        applied_message_delay_ms=(
            max((int(item[0]) for item in delay_reports), default=0)
            / 1000
        ),
    )

def generate_inputs(mode, cfg, query_row=0, npz=None):
    command = (
        f'python3 {PROJECT}/scripts/generate_journal_inputs.py '
        f'--mode {mode} --output-dir {INPUTS} '
        f'--n-records {cfg["n"]} --root-bits {cfg["root_bits"]} '
        f'--block-bits {cfg["block_bits"]} '
        f'--bucket-capacity {cfg["capacity"]} '
        f'--locator-bits {cfg["locator_bits"]} '
        f'--query-row {query_row}'
    )
    if npz is not None:
        command += f' --npz {npz}'
    run(command)
    return json.loads(
        (INPUTS / 'sle_journal_input_metadata.json').read_text()
    )

def run_party(program_name, tag, network='local', extra_env=None):
    log = RUNS / f'{tag}_{network}_run.log'
    execution_env = ENV.copy()
    if extra_env:
        execution_env.update(extra_env)
    output, elapsed, peak = run_monitored(
        f'Scripts/shamir.sh {program_name}',
        cwd=MPSPDZ, env=execution_env, log_path=log,
    )
    return output, elapsed, peak


In [ ]:
# CELL 11. Integrated 512-bit / 128-identity MPC point
integrated_capacity = required_capacity(
    gallery_roots, BIOMETRIC_BLOCK_BITS
)
integrated_cfg = dict(
    n=BIOMETRIC_IDENTITIES,
    root_bits=BIOMETRIC_ROOT_BITS,
    locator_bits=BIOMETRIC_LOCATOR_BITS,
    block_bits=BIOMETRIC_BLOCK_BITS,
    capacity=integrated_capacity,
    padded_slots=(
        (BIOMETRIC_ROOT_BITS // BIOMETRIC_BLOCK_BITS)
        * (2 ** BIOMETRIC_BLOCK_BITS)
        * integrated_capacity
    ),
)
print('Integrated configuration:', integrated_cfg)
if integrated_cfg['padded_slots'] > 80_000:
    raise RuntimeError(
        'The padded index is unexpectedly large. Inspect the roots '
        'or use a fresh runtime before compiling.'
    )

args = (
    f'{integrated_cfg["n"]} {integrated_cfg["root_bits"]} '
    f'{integrated_cfg["block_bits"]} {integrated_cfg["capacity"]} '
    f'{EPOCH} {CLOSED_SET_THRESHOLD} '
    f'{integrated_cfg["locator_bits"]}'
)
program_name, compile_text, compile_seconds = compile_program(
    'sle_integrated_transition_match', args, 'integrated_512x128'
)

integrated_rows = []
for query_row in INTEGRATED_QUERY_ROWS:
    metadata = generate_inputs(
        'integrated', integrated_cfg, query_row=query_row,
        npz=ROOTS_NPZ,
    )
    run_text, wall_seconds, peak_mb = run_party(
        program_name, f'integrated_q{query_row}'
    )
    match = re.search(
        r'mode=integrated .*relation_ok=(\d+) accepted=(\d+) identity=(-?\d+)',
        run_text,
    )
    if match is None:
        raise RuntimeError('Integrated result line was not found')
    relation_ok, accepted, identity = map(int, match.groups())
    if relation_ok != 1 or accepted != 1:
        raise RuntimeError('Integrated transition or identification failed')
    if identity != metadata['expected_identity']:
        raise RuntimeError(
            f'MPC identity {identity} != plaintext {metadata["expected_identity"]}'
        )

    measurement = parse_measurement(
        'sle_integrated', integrated_cfg, query_row,
        compile_text, run_text, compile_seconds,
        wall_seconds, peak_mb,
    )
    measurement.update(
        query_row=query_row,
        relation_ok=relation_ok,
        accepted=accepted,
        mpc_identity=identity,
        expected_identity=metadata['expected_identity'],
        expected_distance=metadata['expected_distance'],
        gallery_sha256=metadata['gallery_sha256'],
        decision_matches_plain=(identity == metadata['expected_identity']),
        decision_matches_true=(identity == int(probe_labels[query_row])),
    )
    integrated_rows.append(measurement)

integrated_results = pd.DataFrame(integrated_rows)
integrated_results.to_csv(
    PROJECT / 'integrated_512bit_128identity.csv', index=False
)
display(integrated_results)


## Point 6: transition scaling through N=128

These rows isolate the cost of the same verified transition contract.
Both methods run in five-party Shamir MP-SPDZ, receive identical secret
inputs, open only `relation_ok`, and retain rollback semantics.

- **SLE:** evolves every equal locator copy with the same hidden PRP.
- **Pointer join:** privately discovers each old-to-new reference with
  an `N x M` equality join.


In [ ]:
# CELL 12. Compile and run N = 8...128 transition measurements
scaling_rows = []
for n in SCALING_N:
    gallery = synthetic_gallery(n, SCALING_ROOT_BITS)
    capacity = required_capacity(gallery, SCALING_BLOCK_BITS)
    cfg = dict(
        n=n,
        root_bits=SCALING_ROOT_BITS,
        locator_bits=SCALING_LOCATOR_BITS,
        block_bits=SCALING_BLOCK_BITS,
        capacity=capacity,
        padded_slots=(
            (SCALING_ROOT_BITS // SCALING_BLOCK_BITS)
            * (2 ** SCALING_BLOCK_BITS)
            * capacity
        ),
    )
    generate_inputs('transition', cfg)

    variants = [('sle', 'sle_transition_optimized')]
    if RUN_POINTER_JOIN:
        variants.append(('pointer_join', 'sle_pointer_join_transition'))

    for mode, source_name in variants:
        if mode == 'sle':
            arguments = (
                f'{n} {SCALING_ROOT_BITS} {SCALING_BLOCK_BITS} '
                f'{capacity} {EPOCH} 0'
            )
        else:
            arguments = (
                f'{n} {SCALING_ROOT_BITS} {SCALING_BLOCK_BITS} '
                f'{capacity} {EPOCH}'
            )
        program, compile_text, compile_seconds = compile_program(
            source_name, arguments, f'scaling_{mode}_n{n}'
        )

        for repeat in range(SCALING_REPEATS):
            # Regenerate identical deterministic inputs after any
            # preceding experiment has replaced Player-Data files.
            generate_inputs('transition', cfg)
            run_text, wall_seconds, peak_mb = run_party(
                program, f'scaling_{mode}_n{n}_r{repeat}'
            )
            row = parse_measurement(
                mode, cfg, repeat, compile_text, run_text,
                compile_seconds, wall_seconds, peak_mb,
            )
            if not row['passed']:
                raise RuntimeError(f'{mode} failed relation verification at N={n}')
            scaling_rows.append(row)

scaling_raw = pd.DataFrame(scaling_rows)
scaling_summary = (
    scaling_raw.groupby(['mode', 'n'], as_index=False)
    .agg(
        total_seconds_mean=('total_seconds', 'mean'),
        total_seconds_std=('total_seconds', 'std'),
        global_mb_mean=('global_mb', 'mean'),
        global_mb_std=('global_mb', 'std'),
        runtime_rounds_mean=('runtime_rounds', 'mean'),
        peak_rss_mb_mean=('peak_rss_mb', 'mean'),
        repeats=('repeat', 'count'),
    )
)

sle_reference = scaling_summary[
    scaling_summary['mode'] == 'sle'
][['n', 'total_seconds_mean', 'global_mb_mean']].rename(columns={
    'total_seconds_mean': 'sle_seconds',
    'global_mb_mean': 'sle_mb',
})
scaling_summary = scaling_summary.merge(sle_reference, on='n')
scaling_summary['runtime_vs_sle'] = (
    scaling_summary['total_seconds_mean'] / scaling_summary['sle_seconds']
)
scaling_summary['communication_vs_sle'] = (
    scaling_summary['global_mb_mean'] / scaling_summary['sle_mb']
)

if 128 not in set(scaling_summary['n']):
    raise RuntimeError('N=128 is missing from the transition table')
if RUN_POINTER_JOIN and len(
    scaling_summary[
        (scaling_summary['n'] == 128)
        & (scaling_summary['mode'] == 'pointer_join')
    ]
) != 1:
    raise RuntimeError('The N=128 pointer-join baseline is missing')

scaling_raw.to_csv(PROJECT / 'transition_scaling_raw.csv', index=False)
scaling_summary.to_csv(
    PROJECT / 'transition_scaling_through_n128.csv', index=False
)
display(scaling_summary)


In [ ]:
# CELL 13. Realistic locator-width sensitivity at fixed N
width_rows = []
gallery = synthetic_gallery(WIDTH_TEST_N, SCALING_ROOT_BITS)
capacity = required_capacity(gallery, SCALING_BLOCK_BITS)

for locator_bits in LOCATOR_WIDTHS:
    cfg = dict(
        n=WIDTH_TEST_N,
        root_bits=SCALING_ROOT_BITS,
        locator_bits=locator_bits,
        block_bits=SCALING_BLOCK_BITS,
        capacity=capacity,
        padded_slots=(
            (SCALING_ROOT_BITS // SCALING_BLOCK_BITS)
            * (2 ** SCALING_BLOCK_BITS)
            * capacity
        ),
    )
    for mode, source_name in [
        ('sle', 'sle_transition_optimized'),
        ('pointer_join', 'sle_pointer_join_transition'),
    ]:
        if mode == 'sle':
            arguments = (
                f'{WIDTH_TEST_N} {SCALING_ROOT_BITS} '
                f'{SCALING_BLOCK_BITS} {capacity} {EPOCH} 0'
            )
        else:
            arguments = (
                f'{WIDTH_TEST_N} {SCALING_ROOT_BITS} '
                f'{SCALING_BLOCK_BITS} {capacity} {EPOCH}'
            )
        program, compile_text, compile_seconds = compile_program(
            source_name, arguments,
            f'width_{mode}_{locator_bits}bit'
        )
        for repeat in range(WIDTH_REPEATS):
            generate_inputs('transition', cfg)
            run_text, wall_seconds, peak_mb = run_party(
                program,
                f'width_{mode}_{locator_bits}bit_r{repeat}'
            )
            row = parse_measurement(
                mode, cfg, repeat, compile_text, run_text,
                compile_seconds, wall_seconds, peak_mb,
            )
            if not row['passed']:
                raise RuntimeError(
                    f'{mode} failed with {locator_bits}-bit locators'
                )
            width_rows.append(row)

width_raw = pd.DataFrame(width_rows)
width_summary = (
    width_raw.groupby(['mode', 'locator_bits'], as_index=False)
    .agg(
        total_seconds_mean=('total_seconds', 'mean'),
        total_seconds_std=('total_seconds', 'std'),
        global_mb_mean=('global_mb', 'mean'),
        runtime_rounds_mean=('runtime_rounds', 'mean'),
        repeats=('repeat', 'count'),
    )
)
if 64 not in set(width_summary['locator_bits']):
    raise RuntimeError('The required 64-bit locator point is missing')
width_raw.to_csv(PROJECT / 'locator_width_raw.csv', index=False)
width_summary.to_csv(
    PROJECT / 'locator_width_32_vs_64.csv', index=False
)
display(width_summary)


## Point 3: emulated WAN

The next cell first measures the two transition methods locally, then
repeats the exact executions under a 20 ms latency setting. Linux
`tc/netem` is preferred. If the Colab container does not expose
`CAP_NET_ADMIN`, latency is injected at MP-SPDZ's logical SSL-message
path, with per-process message counters printed into the audit logs.

End-to-end wall-clock time is the primary WAN metric. The point is
accepted only when all relation checks pass, all five parties report
delayed traffic, WAN wall time increases by at least 20%, and the
relative order of SLE and pointer join remains unchanged. This is a
controlled single-host WAN emulation, not a multi-region deployment.


In [ ]:
# CELL 14. Local versus validated emulated-WAN measurements
def set_netem(delay_ms):
    command = [
        'tc', 'qdisc', 'replace', 'dev', 'lo', 'root', 'netem',
        'delay', f'{delay_ms}ms'
    ]
    result = subprocess.run(command, text=True, capture_output=True)
    return result.returncode == 0, result.stdout + result.stderr

def clear_netem():
    subprocess.run(
        ['tc', 'qdisc', 'del', 'dev', 'lo', 'root'],
        text=True, capture_output=True,
    )

wan_gallery = synthetic_gallery(WAN_N, SCALING_ROOT_BITS)
wan_capacity = required_capacity(wan_gallery, SCALING_BLOCK_BITS)
wan_cfg = dict(
    n=WAN_N,
    root_bits=SCALING_ROOT_BITS,
    locator_bits=64,
    block_bits=SCALING_BLOCK_BITS,
    capacity=wan_capacity,
    padded_slots=(
        (SCALING_ROOT_BITS // SCALING_BLOCK_BITS)
        * (2 ** SCALING_BLOCK_BITS)
        * wan_capacity
    ),
)

wan_programs = {}
for mode, source_name in [
    ('sle', 'sle_transition_optimized'),
    ('pointer_join', 'sle_pointer_join_transition'),
]:
    if mode == 'sle':
        arguments = (
            f'{WAN_N} {SCALING_ROOT_BITS} {SCALING_BLOCK_BITS} '
            f'{wan_capacity} {EPOCH} 0'
        )
    else:
        arguments = (
            f'{WAN_N} {SCALING_ROOT_BITS} {SCALING_BLOCK_BITS} '
            f'{wan_capacity} {EPOCH}'
        )
    wan_programs[mode] = (
        *compile_program(source_name, arguments, f'wan_{mode}'),
    )

wan_rows = []
clear_netem()
for mode, (program, compile_text, compile_seconds) in wan_programs.items():
    for repeat in range(WAN_REPEATS):
        generate_inputs('transition', wan_cfg)
        run_text, wall_seconds, peak_mb = run_party(
            program, f'wan_{mode}_r{repeat}', network='local'
        )
        wan_rows.append(parse_measurement(
            mode, wan_cfg, repeat, compile_text, run_text,
            compile_seconds, wall_seconds, peak_mb, network='local'
        ))

WAN_EMPIRICAL = False
WAN_BACKEND = 'disabled'
NETEM_MESSAGE = 'WAN test disabled by configuration'
if RUN_WAN:
    ok, message = set_netem(WAN_ONE_WAY_DELAY_MS)
    NETEM_MESSAGE = message.strip()
    if ok:
        WAN_EMPIRICAL = True
        WAN_BACKEND = 'tc_netem'
        try:
            ping_text, _, _ = run('ping -c 3 -q 127.0.0.1')
            ping_match = re.search(
                r'= [0-9.]+/([0-9.]+)/[0-9.]+/[0-9.]+ ms', ping_text
            )
            WAN_PING_RTT_MS = (
                float(ping_match.group(1)) if ping_match else np.nan
            )
            for mode, (program, compile_text, compile_seconds) in wan_programs.items():
                for repeat in range(WAN_REPEATS):
                    generate_inputs('transition', wan_cfg)
                    run_text, wall_seconds, peak_mb = run_party(
                        program, f'wan_{mode}_r{repeat}',
                        network='netem_wan'
                    )
                    wan_rows.append(parse_measurement(
                        mode, wan_cfg, repeat, compile_text, run_text,
                        compile_seconds, wall_seconds, peak_mb,
                        network='netem_wan'
                    ))
        finally:
            clear_netem()
    else:
        WAN_PING_RTT_MS = np.nan
        print('NETEM UNAVAILABLE:', NETEM_MESSAGE)
        if not USERSPACE_DELAY_PATCHED:
            print('User-space delay hook is also unavailable.')
        else:
            WAN_EMPIRICAL = True
            WAN_BACKEND = 'userspace_logical_message_delay'
            delay_env = {
                'SLE_WAN_MESSAGE_DELAY_US': str(
                    int(WAN_ONE_WAY_DELAY_MS * 1000)
                )
            }
            for mode, (program, compile_text, compile_seconds) in wan_programs.items():
                for repeat in range(WAN_REPEATS):
                    generate_inputs('transition', wan_cfg)
                    run_text, wall_seconds, peak_mb = run_party(
                        program, f'wan_{mode}_r{repeat}',
                        network='userspace_wan', extra_env=delay_env,
                    )
                    wan_rows.append(parse_measurement(
                        mode, wan_cfg, repeat, compile_text, run_text,
                        compile_seconds, wall_seconds, peak_mb,
                        network='userspace_wan',
                    ))
else:
    WAN_PING_RTT_MS = np.nan

wan_raw = pd.DataFrame(wan_rows)
wan_summary = (
    wan_raw.groupby(['network', 'mode'], as_index=False)
    .agg(
        wall_seconds_mean=('wall_seconds', 'mean'),
        wall_seconds_std=('wall_seconds', 'std'),
        total_seconds_mean=('total_seconds', 'mean'),
        global_mb_mean=('global_mb', 'mean'),
        runtime_rounds_mean=('runtime_rounds', 'mean'),
        delay_report_count=('delay_report_count', 'sum'),
        delayed_messages=('delayed_messages', 'sum'),
        delayed_bytes=('delayed_bytes', 'sum'),
        repeats=('repeat', 'count'),
    )
)

# WAN comparisons must use end-to-end wall time. MP-SPDZ's internal
# timer can exclude transport waiting and is therefore retained only
# as a secondary diagnostic column.
local_order = (
    wan_summary[wan_summary['network'] == 'local']
    .sort_values('wall_seconds_mean')['mode'].tolist()
)
wan_network_label = (
    'netem_wan' if WAN_BACKEND == 'tc_netem' else 'userspace_wan'
)
wan_order = (
    wan_summary[wan_summary['network'] == wan_network_label]
    .sort_values('wall_seconds_mean')['mode'].tolist()
)
WAN_ORDER_PRESERVED = WAN_EMPIRICAL and local_order == wan_order

local_wall = {
    row['mode']: float(row['wall_seconds_mean'])
    for _, row in wan_summary[
        wan_summary['network'] == 'local'
    ].iterrows()
}
wan_wall = {
    row['mode']: float(row['wall_seconds_mean'])
    for _, row in wan_summary[
        wan_summary['network'] == wan_network_label
    ].iterrows()
}
WAN_WALL_SLOWDOWNS = {
    mode: float(wan_wall[mode] / local_wall[mode])
    for mode in sorted(set(local_wall) & set(wan_wall))
}
LOCAL_POINTER_JOIN_OVER_SLE = (
    float(local_wall['pointer_join'] / local_wall['sle'])
    if {'sle', 'pointer_join'} <= set(local_wall) else None
)
WAN_POINTER_JOIN_OVER_SLE = (
    float(wan_wall['pointer_join'] / wan_wall['sle'])
    if {'sle', 'pointer_join'} <= set(wan_wall) else None
)
wan_summary['wall_slowdown_vs_local'] = wan_summary.apply(
    lambda row: (
        row['wall_seconds_mean'] / local_wall[row['mode']]
        if row['network'] != 'local' and row['mode'] in local_wall
        else 1.0
    ),
    axis=1,
)

wan_rows_only = wan_raw[wan_raw['network'] == wan_network_label]
WAN_ALL_RELATIONS_PASS = bool(
    len(wan_rows_only) and wan_rows_only['passed'].all()
)
WAN_REPEATS_COMPLETE = bool(
    len(wan_rows_only)
    and wan_rows_only.groupby('mode').size().min() >= WAN_REPEATS
)
WAN_DELAY_EFFECT_OBSERVED = bool(
    len(WAN_WALL_SLOWDOWNS) == 2
    and min(WAN_WALL_SLOWDOWNS.values()) >= WAN_MIN_WALL_SLOWDOWN
)
if WAN_BACKEND == 'tc_netem':
    WAN_TRAFFIC_DELAYED = bool(
        np.isfinite(WAN_PING_RTT_MS)
        and WAN_PING_RTT_MS >= 1.5 * WAN_ONE_WAY_DELAY_MS
    )
elif WAN_BACKEND == 'userspace_logical_message_delay':
    # Scripts/shamir.sh launches all parties but forwards the detailed
    # benchmark stream only for party 0. One report per captured run is
    # therefore expected; the common binary and environment variable
    # apply the same delay hook to every launched party.
    WAN_TRAFFIC_DELAYED = bool(
        len(wan_rows_only)
        and wan_rows_only['delay_report_count'].ge(1).all()
        and wan_rows_only['delayed_messages'].gt(0).all()
        and wan_rows_only['applied_message_delay_ms'].eq(
            WAN_ONE_WAY_DELAY_MS
        ).all()
    )
else:
    WAN_TRAFFIC_DELAYED = False

WAN_VALIDATED = all([
    WAN_EMPIRICAL,
    WAN_ALL_RELATIONS_PASS,
    WAN_REPEATS_COMPLETE,
    WAN_TRAFFIC_DELAYED,
    WAN_DELAY_EFFECT_OBSERVED,
    WAN_ORDER_PRESERVED,
])

wan_status = {
    'empirical': WAN_EMPIRICAL,
    'validated': WAN_VALIDATED,
    'backend': WAN_BACKEND,
    'configured_delay_ms': WAN_ONE_WAY_DELAY_MS,
    'netem_one_way_delay_ms': (
        WAN_ONE_WAY_DELAY_MS if WAN_BACKEND == 'tc_netem' else None
    ),
    'userspace_logical_message_delay_ms': (
        WAN_ONE_WAY_DELAY_MS
        if WAN_BACKEND == 'userspace_logical_message_delay' else None
    ),
    'measured_ping_rtt_ms': (
        float(WAN_PING_RTT_MS)
        if np.isfinite(WAN_PING_RTT_MS) else None
    ),
    'wall_slowdown_by_method': WAN_WALL_SLOWDOWNS,
    'pointer_join_over_sle_local': LOCAL_POINTER_JOIN_OVER_SLE,
    'pointer_join_over_sle_wan': WAN_POINTER_JOIN_OVER_SLE,
    'all_relations_pass': WAN_ALL_RELATIONS_PASS,
    'repeats_complete': WAN_REPEATS_COMPLETE,
    'delay_injection_verified': WAN_TRAFFIC_DELAYED,
    'captured_delay_report_scope': (
        'party_0_stdout; common hook applied to all launched parties'
        if WAN_BACKEND == 'userspace_logical_message_delay' else None
    ),
    'wall_delay_effect_observed': WAN_DELAY_EFFECT_OBSERVED,
    'local_order': local_order,
    'wan_order': wan_order,
    'ordering_preserved': WAN_ORDER_PRESERVED,
    'netem_message': NETEM_MESSAGE,
}
(PROJECT / 'wan_status.json').write_text(
    json.dumps(wan_status, indent=2), encoding='utf-8'
)
wan_raw.to_csv(PROJECT / 'wan_emulation_raw.csv', index=False)
wan_summary.to_csv(PROJECT / 'wan_emulation_summary.csv', index=False)
display(wan_summary)
print(json.dumps(wan_status, indent=2))


In [ ]:
# CELL 15. Journal-style figures and compact LaTeX tables
plt.rcParams.update({
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'legend.fontsize': 8,
    'figure.dpi': 140,
})

fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.25))
styles = {
    'sle': dict(color='#222222', marker='o', label='SLE'),
    'pointer_join': dict(color='#0072B2', marker='s', label='Pointer join'),
}
for mode, group in scaling_summary.groupby('mode'):
    group = group.sort_values('n')
    axes[0].plot(
        group['n'], group['total_seconds_mean'], linewidth=1.6,
        markersize=5, **styles[mode]
    )
    axes[1].plot(
        group['n'], group['global_mb_mean'], linewidth=1.6,
        markersize=5, **styles[mode]
    )
axes[0].set_title('(a) Transition runtime')
axes[0].set_xlabel('Enrolled identities, N')
axes[0].set_ylabel('Total runtime (s)')
axes[1].set_title('(b) Communication')
axes[1].set_xlabel('Enrolled identities, N')
axes[1].set_ylabel('Global communication (MB)')

if WAN_VALIDATED:
    pivot = wan_summary.pivot(
        index='mode', columns='network', values='wall_seconds_mean'
    ).loc[['sle', 'pointer_join']]
    wan_column = (
        'netem_wan' if WAN_BACKEND == 'tc_netem'
        else 'userspace_wan'
    )
    x = np.arange(len(pivot))
    axes[2].bar(x - 0.18, pivot['local'], 0.36, color='#777777', label='Local')
    axes[2].bar(x + 0.18, pivot[wan_column], 0.36, color='#D55E00', label='WAN')
    axes[2].set_xticks(x, ['SLE', 'Pointer join'])
    axes[2].set_ylabel('End-to-end wall time (s)')
    axes[2].legend(frameon=False)
else:
    axes[2].text(
        0.5, 0.5, 'WAN validation failed\ninspect wan_status.json',
        ha='center', va='center', transform=axes[2].transAxes
    )
    axes[2].set_xticks([])
    axes[2].set_yticks([])
axes[2].set_title('(c) Local versus emulated WAN')

for axis in axes[:2]:
    axis.grid(True, linestyle=':', linewidth=0.7, alpha=0.6)
axes[0].legend(frameon=False)
fig.tight_layout()
figure_path = PROJECT / 'journal_transition_and_wan.png'
fig.savefig(figure_path, dpi=300, bbox_inches='tight')
plt.show()

scaling_table = scaling_summary[[
    'mode', 'n', 'total_seconds_mean', 'global_mb_mean',
    'runtime_rounds_mean', 'runtime_vs_sle',
    'communication_vs_sle',
]]
width_table = width_summary.copy()
wan_table = wan_summary[[
    'network', 'mode', 'wall_seconds_mean', 'wall_seconds_std',
    'wall_slowdown_vs_local', 'global_mb_mean',
    'runtime_rounds_mean', 'repeats',
]].copy()
(PROJECT / 'table_transition_n128.tex').write_text(
    scaling_table.to_latex(index=False, float_format='%.3f'),
    encoding='utf-8',
)
(PROJECT / 'table_locator_width.tex').write_text(
    width_table.to_latex(index=False, float_format='%.3f'),
    encoding='utf-8',
)
(PROJECT / 'table_wan.tex').write_text(
    wan_table.to_latex(index=False, float_format='%.3f'),
    encoding='utf-8',
)
print('Saved figure:', figure_path)


In [ ]:
# CELL 16. Automatic readiness checks and result bundle
checks = {
    'integrated_uses_128_identities': BIOMETRIC_IDENTITIES == 128,
    'integrated_uses_512bit_roots': BIOMETRIC_ROOT_BITS == 512,
    'integrated_uses_64bit_locators': BIOMETRIC_LOCATOR_BITS == 64,
    'all_integrated_relations_pass': bool(integrated_results['relation_ok'].eq(1).all()),
    'all_mpc_decisions_match_plain': bool(integrated_results['decision_matches_plain'].all()),
    'transition_table_contains_n128': 128 in set(scaling_summary['n']),
    'n128_contains_both_methods': set(
        scaling_summary[scaling_summary['n'] == 128]['mode']
    ) == {'sle', 'pointer_join'},
    'realistic_64bit_locator_point_present': 64 in set(width_summary['locator_bits']),
    'wan_is_empirical': bool(WAN_EMPIRICAL),
    'wan_emulated_delay_verified': bool(WAN_TRAFFIC_DELAYED),
    'wan_wall_time_increased': bool(WAN_DELAY_EFFECT_OBSERVED),
    'wan_repeats_complete': bool(WAN_REPEATS_COMPLETE),
    'wan_all_relations_pass': bool(WAN_ALL_RELATIONS_PASS),
    'wan_method_order_preserved': bool(WAN_ORDER_PRESERVED),
    'wan_point_fully_validated': bool(WAN_VALIDATED),
    'not_synthetic_biometric_data': not USE_SYNTHETIC_BIOMETRIC_SMOKE,
}
readiness = pd.DataFrame([
    {'check': key, 'passed': value} for key, value in checks.items()
])
readiness.to_csv(PROJECT / 'journal_readiness_checks.csv', index=False)
display(readiness)

manifest = {
    'generated_at_utc': pd.Timestamp.utcnow().isoformat(),
    'run_profile': RUN_PROFILE,
    'mp_spdz_commit': MP_SPDZ_COMMIT,
    'players': PLAYERS,
    'threshold': THRESHOLD,
    'biometric_configuration': integrated_cfg,
    'scaling_n': SCALING_N,
    'scaling_repeats': SCALING_REPEATS,
    'locator_widths': LOCATOR_WIDTHS,
    'wan_status': wan_status,
    'checks': checks,
    'claim_boundary': (
        f'The WAN point is a controlled single-host {WAN_BACKEND} '
        'experiment evaluated with end-to-end wall-clock time; it is '
        'not a geographically distributed deployment. The N-scaling '
        'rows isolate the verified transition contract and are not '
        'cross-paper SOTA query timings.'
    ),
}
(PROJECT / 'reproducibility_manifest.json').write_text(
    json.dumps(manifest, indent=2), encoding='utf-8'
)

bundle = PROJECT / 'sle_journal_technical_validation_results.zip'
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in PROJECT.rglob('*'):
        if not path.is_file() or path == bundle:
            continue
        if 'runs' in path.parts and path.suffix == '.log':
            # Keep logs for audit, but omit very large compiler logs.
            if path.stat().st_size > 5_000_000:
                continue
        archive.write(path, arcname=path.relative_to(PROJECT))

print('Saved:', bundle)
failed = readiness[~readiness['passed']]
if len(failed):
    print('Attention: these journal checks did not pass:')
    display(failed)
else:
    print('All requested technical checks passed.')

try:
    from google.colab import files
    files.download(str(bundle))
except ImportError:
    print('Automatic download is available only in Colab.')
